In [1]:
import torch
from PIL import Image
import gradio as gr
from diffusers import StableDiffusionPipeline, StableDiffusionDepth2ImgPipeline

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


In [2]:

t2i_device= torch.device("cuda:0")
pipe_t2i= StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16
).to(t2i_device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

StableDiffusionSafetyChecker LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/safety_checker
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
vision_model.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
def text_to_image(text_prompt, steps=30, guidance_scale=7.5):
    if not text_prompt or not text_prompt.strip():
        raise ValueError("Please you need to enter a prompt.")
    image=pipe_t2i(
        text_prompt,
        num_inference_steps=steps,
        guidance_scale=guidance_scale
    ).images[0]

    torch.cuda.empty_cache()
    return image

In [10]:
depth_device=torch.device("cuda:0")
pipe_depth2img= StableDiffusionDepth2ImgPipeline.from_pretrained(
    "sd2-community/stable-diffusion-2-depth",
    torch_dtype=torch.float16
).to(depth_device)

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--sd2-community--stable-diffusion-2-depth/snapshots/6cb92dd9430a7f6da8d9e99d7b60acdebcc348b7/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/414 [00:00<?, ?it/s]

In [11]:
def depth_to_image(image, prompt, steps=30, guidance_scale=7.5):
    with torch.cuda.device(depth_device):
        output=pipe_depth2img(
            prompt=prompt,
            image=image,
            num_inference_steps=steps,
            guidance_scale=guidance_scale
        )
        image_out = output.images[0]
    torch.cuda.empty_cache()
    return image_out

In [15]:
with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="indigo",
        secondary_hue="blue",
        radius_size="lg"
    ),
    title="Stable Diffusion Studio"
) as demo:

    gr.HTML("""
        <div style="text-align:center; padding: 20px">
            <h1>Stable Diffusion Studio</h1>
        </div>
    """)

    with gr.Tabs():
        with gr.TabItem("Text to Image"):

            with gr.Row():

                with gr.Column(scale=1):
                    gr.Markdown("### The Prompt")

                    prompt_input = gr.Textbox(
                        placeholder="Type a text prompt to generate an image...",
                        lines=4,
                        show_label=False
                    )

                    steps_slider = gr.Slider(
                        1, 50,
                        value=30,
                        label="Sampling Steps"
                    )

                    guidance_slider = gr.Slider(
                        1, 20,
                        value=7.5,
                        label="Guidance Scale"
                    )

                    generate_btn = gr.Button(
                        "Generate Image",
                        variant="primary",
                        size="lg"
                    )

                with gr.Column(scale=1):
                    gr.Markdown("### the Result")
                    output_img = gr.Image(
                        label="Generated Image",
                        height=500
                    )

            generate_btn.click(
                fn=text_to_image,
                inputs=[prompt_input, steps_slider, guidance_slider],
                outputs=output_img
            )

        with gr.TabItem("Image to Depth Restyle"):

            with gr.Row():

                with gr.Column(scale=1):

                    gr.Markdown("### Upload an Image")
                    img_input = gr.Image(type="pil")

                    gr.Markdown("### Transformation Prompt")
                    prompt_input2 = gr.Textbox(
                        placeholder="Enter a prompt to restyle the image...",
                        lines=3,
                        show_label=False
                    )

                    steps_slider2 = gr.Slider(
                        1, 50,
                        value=25,
                        label="Sampling Steps"
                    )

                    guidance_slider2 = gr.Slider(
                        1, 20,
                        value=7.5,
                        label="Guidance Scale"
                    )

                    restyle_btn = gr.Button(
                        "Restyle Image",
                        variant="primary",
                        size="lg"
                    )

                with gr.Column(scale=1):
                    gr.Markdown("### The Result")
                    output_img2 = gr.Image(
                        label="Depth Restyled Image",
                        height=500
                    )

            restyle_btn.click(
                fn=depth_to_image,
                inputs=[img_input, prompt_input2, steps_slider2, guidance_slider2],
                outputs=output_img2
            )

    gr.HTML("""
        <div style="text-align:center; padding: 15px; opacity:0.6">
    """)

demo.launch()

/tmp/ipykernel_385/2382417294.py:1: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ed2325dec7e8009053.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [7]:
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fef6c9bd997f8e688d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
